# Rainfall erosivity calculating tool based on IMERG V06 and Machine learning
## Latitude and Longitude point data

**Authors:**

<ul style="line-height:1.5;">
<li>Ameng Zou <a href="mailto:amengzou@arizona.edu">(amengzou@arizona.edu)</a></li>
<li>Shang Gao <a href="mailto:shanggao@arizona.edu">(shanggao@arizona.edu)</a></li>
</ul>

**Last updated: 08/06/2025**

**Purpose:**

This notebook provides code examples for calculating rainfall erosivity derived from IMERG precipittation data which makes it available to acquire rainfall erosivity value (R-factor) anywhere globally. For getting more accurate rainfall eorsivity, this notebook also applies a machine learning algorithm to reliably correct the rainfall erosivity value from satellite data which helps user to collect high-accuracy rainfall erosivity data.

**Audience:**

Researchers who are familiar with Jupyter Notebooks, basic Python, and basic hydrologic data analysis.

**Description:**

This notebook takes as inputs A set of latitude and longitude coordinates. Firstly, it collect original sattlite rainfall erosivity data from Emberson dataset. It then retrieves data from Google Earth Engine Assets, applies machine learning algorithm to predict the error between the ture value and originial rainfall erosivity data, saves the data as a comma separated variable (CSV) file, and visualize it.

**Data Description:**

Original rainfall erosivity data is from Erosivity_IMERGV06B_30min_2001_2021.tif.

**Software Requirements:**

This notebook uses the following libraries, and some of these libraries may have their own additional dependencies:

> pandas            2.0.2  
> numpy             1.24.3  
> matplotlib        3.6.3  
> rasterio          1.4.3  
> earthengine-api   1.5.23  
> pycaret           3.3.2  
> cartopy           0.24.1  
> tqdm              4.66.5

If the requiring libraries are already installed where you are working, then you can proceed to execute the notebook. If however you encounter a problem with missing libraries, you can install them before you execute the notebook:

In [ ]:
import sys
!"{sys.executable}" -m pip install pandas
!"{sys.executable}" -m pip install numpy
!"{sys.executable}" -m pip install matplotlib
!"{sys.executable}" -m pip install rasterio
!"{sys.executable}" -m pip install earthengine-api
!"{sys.executable}" -m pip install pycaret
!"{sys.executable}" -m pip install cartopy
!"{sys.executable}" -m pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


### 1. Import libraries

Import the libraries needed to run this notebook:

In [ ]:
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.image as mpimg
from rasterio.transform import rowcol
from tqdm import tqdm
from pycaret.regression import load_model, predict_model
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, Normalize

### 2. Rainfall erosivity estimates derived from IMERG V06

Extract R factor from Erosivity_IMERGV06B_30min_2001_2021.tif (Emberson, 2023) according to the latitude (lat) and longitude (lon) added by the user in Inputs. csv.
Reference:
Emberson, R. A. (2023). Dynamic rainfall erosivity estimates derived from IMERG data. Hydrology and Earth System Sciences, 27(19), 3547-3563.

In [ ]:
# Input file paths
csv_path = r'C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Inputs.csv'  # CSV must contain 'lat' and 'lon' columns
tif_path = r'C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Erosivity_IMERGV06B_30min_2001_2021.tif'

# Read the input CSV
df = pd.read_csv(csv_path)

# Open the GeoTIFF file
with rasterio.open(tif_path) as src:
    transform = src.transform
    band = src.read(1)  # Read the first band
    nodata = src.nodata

    # Prepare a list to store extracted values
    erosivity_values = []

    # Iterate over each coordinate
    for _, row in tqdm(df.iterrows(), total=len(df)):
        lat, lon = row['lat'], row['lon']
        try:
            # Convert lat/lon to row/column indices
            r, c = rowcol(transform, lon, lat)
            value = band[r, c]

            # Handle NoData values
            if value == nodata:
                value = None
        except:
            value = None

        erosivity_values.append(value)

# Add the extracted values as a new column
df['Erosivity_IMERG_V06'] = erosivity_values

# Save the result to a new CSV file
output_csv = 'Outputs.csv'
df.to_csv(output_csv, index=False)
print(f'Extraction completed. Saved to {output_csv}')

100%|█████████████████████████████████████████████████████████████████████████████| 137/137 [00:00<00:00, 19568.85it/s]

Extraction completed. Saved to Outputs.csv


### 3. Error prediction for rainfall erosivity

Apply a pre-trained machine learning model and 23 environmental variables from Google Earth Engine (GEE) for each location. Predicted errors are appended as a new column 'Error_Prediction' in the original DataFrame, which is then saved back to CSV.

In [ ]:
# Input CSV file with 'lat' and 'lon' columns
csv_path = r'C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Outputs.csv'

# Output path
output_csv = csv_path

# Path to the trained PyCaret model
model_path = r'C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Second_Round_Optimized_ET_Model_Emberson'

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-amengzou')

# Load the trained model
model = load_model(model_path)

# === LOAD GEE LAYERS ===

def load_gee_images():
    return ee.Image('projects/ee-amengzou/assets/Global_IMERG_Monthly_Mean') \
        .addBands([
            ee.Image('projects/ee-amengzou/assets/Global_MODIS13_NDVI_Mean'),
            ee.Image('projects/ee-amengzou/assets/Global_SRTM_Elevation'),
            ee.Image('projects/ee-amengzou/assets/Global_MODIS11').select('LST_Day_1km'),
            ee.Image('projects/ee-amengzou/assets/Global_MODIS11').select('LST_Night_1km'),
            ee.Image('projects/ee-amengzou/assets/Global_MODIS11').select('Emis_31'),
            ee.Image('projects/ee-amengzou/assets/Global_MODIS11').select('Emis_32'),
            ee.Image('projects/ee-amengzou/assets/Global_GLDAS_Variables_Mean').select([
                'AvgSurfT_inst', 'Tair_f_inst', 'Qair_f_inst', 'Albedo_inst', 'Evap_tavg',
                'PotEvap_tavg', 'LWdown_f_tavg', 'Lwnet_tavg', 'Qg_tavg', 'RootMoist_inst',
                'SWdown_f_tavg', 'Swnet_tavg'
            ]),
            ee.Image('projects/ee-amengzou/assets/Global_ERA5_Rainfall_Intermittency_Mean'),
            ee.Image('projects/ee-amengzou/assets/Global_ERA5_R10'),
            ee.Image('projects/ee-amengzou/assets/Global_ERA5_R95P'),
            ee.Image('projects/ee-amengzou/assets/Global_ERA5_R99P')
        ])

gee_image = load_gee_images()

# Band name mapping from GEE to model input
rename_map = {
    'precipitation': 'Precipitation', 'ndvi': 'NDVI', 'elevation': 'Elevation',
    'lst_day_1km': 'LST_Day_1km', 'lst_night_1km': 'LST_Night_1km',
    'emis_31': 'Emis_31', 'emis_32': 'Emis_32', 'avgsurft_inst': 'AvgSurfT_inst',
    'tair_f_inst': 'Tair_f_inst', 'qair_f_inst': 'Qair_f_inst', 'albedo_inst': 'Albedo_inst',
    'evap_tavg': 'Evap_tavg', 'potevap_tavg': 'PotEvap_tavg', 'lwdown_f_tavg': 'LWdown_f_tavg',
    'lwnet_tavg': 'Lwnet_tavg', 'qg_tavg': 'Qg_tavg', 'rootmoist_inst': 'RootMoist_inst',
    'swdown_f_tavg': 'SWdown_f_tavg', 'swnet_tavg': 'Swnet_tavg',
    'Rainfall_Intermittency': 'Rainfall_intermittency', 'r10': 'R10', 'r95p': 'R95P', 'r99p': 'R99P'
}
model_features = list(rename_map.values())

# === RUN PREDICTION ===

# Load input CSV
df = pd.read_csv(csv_path)
predictions = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Predicting"):
    lat = row['lat']
    lon = row['lon']
    point = ee.Geometry.Point([lon, lat])
    sample = gee_image.sample(region=point.buffer(5000).bounds(), scale=1000, numPixels=1).first()

    try:
        sample_info = sample.getInfo()
        if sample_info is None or 'properties' not in sample_info:
            predictions.append(None)
            continue

        props = sample_info['properties']
        props_df = pd.DataFrame([props])
        props_df.rename(columns=rename_map, inplace=True)

        # Check if all required features are present
        if not set(model_features).issubset(props_df.columns):
            predictions.append(None)
            continue

        # Predict
        props_df = props_df[model_features]
        result = predict_model(model, data=props_df)
        predictions.append(result['prediction_label'].iloc[0])

    except Exception as e:
        print(f"⚠️ Failed at ({lat}, {lon}): {e}")
        predictions.append(None)

# Add predictions as a new column with desired name
df['Error_Prediction'] = predictions

# Save result
df.to_csv(output_csv, index=False)
print(f'Prediction completed. Saved to {output_csv}')

Transformation Pipeline and Model Successfully Loaded


Predicting: 100%|████████████████████████████████████████████████████████████████████| 137/137 [00:55<00:00,  2.46it/s]

Prediction completed. Saved to C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Outputs.csv


### 4. Rainfall erosivity correction

Rainfall erosivity (corrected) = 'Rainfall erosivity estimates derived from IMERG V06 (Emberson)' + 'Rainfall erosivity estimates error prediction'

In [ ]:
# File path
csv_path = r'C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Outputs.csv'

# Read the data
df = pd.read_csv(csv_path)

# Ensure column names are correct
erosivity_col = 'Erosivity_IMERG_V06'
error_col = 'Error_Prediction'

# Check if required columns exist
if erosivity_col in df.columns and error_col in df.columns:
    # Calculate corrected erosivity
    df['Corrected_Erosivity'] = df[erosivity_col] - df[error_col]

    # Save back to file
    df.to_csv(csv_path, index=False)
    print(f"✅ Corrected Erosivity column added and saved to {csv_path}")
else:
    print("❌ Column names not found. Please check that the columns exist in the CSV file.")

✅ Corrected Erosivity column added and saved to C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Outputs.csv


### 5. Plot the Data (Optional)

Visualize the spatial distribution of machine learning–corrected rainfall erosivity. This setup helps users visually evaluate the spatial patterns in the corrected predictions.

In [ ]:
# Set global font to Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'

# 1. Load data
csv_path = r'C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Outputs.csv'
df = pd.read_csv(csv_path)

# 2. Check required columns
required_cols = ['lat', 'lon', 'Corrected_Erosivity']
if not all(col in df.columns for col in required_cols):
    raise ValueError(f"Missing required columns: {required_cols}")

# 3. Pivot data for gridded display
pivot_map = df.pivot(index='lat', columns='lon', values='Corrected_Erosivity')
lats = pivot_map.index.values
lons = pivot_map.columns.values
corrected_erosivity_map = pivot_map.values

# 4. Map extent (from data)
lat_min, lat_max = lats.min(), lats.max()
lon_min, lon_max = lons.min(), lons.max()

# 5. Color map and normalization
cmap = LinearSegmentedColormap.from_list("custom", ["blue", "cyan", "yellow", "red"])
vmin = np.nanmin(corrected_erosivity_map)
vmax = np.nanmax(corrected_erosivity_map)

if vmin < 0 and vmax > 0:
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
else:
    norm = Normalize(vmin=vmin, vmax=vmax)

# 6. Plotting
fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

# Add map features
ax.coastlines()
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.xlabel_style = {'size': 14, 'fontname': 'Times New Roman'}
gl.ylabel_style = {'size': 14, 'fontname': 'Times New Roman'}

# Draw the corrected erosivity map
# Scatter plot for station-based corrected erosivity
mesh = ax.scatter(df['lon'], df['lat'], c=df['Corrected_Erosivity'],
                  cmap=cmap, norm=norm, s=100, edgecolor='k', linewidth=0.5,
                  transform=ccrs.PlateCarree())

# Add colorbar
cbar = plt.colorbar(mesh, orientation='horizontal', pad=0.05, shrink=0.8, extend='both')
cbar.set_label('Corrected Erosivity (MJ·mm·ha⁻¹·h⁻¹·yr⁻¹)', fontsize=16)
cbar.ax.tick_params(labelsize=12)

# Add title
ax.set_title('Spatial Distribution of Corrected Rainfall Erosivity', fontsize=18, pad=20)

# Save and show
output_path = r'C:\Users\amengzou\Desktop\2025_Fall\Research_Rainfall_Erosivity_Ryan_Stations\Jupyter_tool\Corrected_Erosivity_Map.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

ValueError: Index contains duplicate entries, cannot reshape